CROSSVALIDATION BW THRESHOLD


In [30]:
import os
import pandas as pd
#import plotly.express as px
import cv2 as cv
import argparse
import numpy as np
import time
import statistics
from datetime import datetime
from matplotlib import pyplot as plt
import seaborn as sns
from scipy.stats import gaussian_kde
from scipy.signal import find_peaks
import sklearn.metrics



In [31]:
#ruta_carpeta = r'C:\Users\inges\OneDrive - UDIT\src\photoneu\dataset\deeplabcut\labeled-data-ordered'
ruta_carpeta = r'C:\Users\inges\OneDrive - UDIT\src\photoneu\dataset\MovAl.v1i.tensorflow\train'

#ruta_carpeta = r'C:\Users\inges\OneDrive - UDIT\src\photoneu\dataset\movai_3_mice'
kernel_3 = cv.getStructuringElement(cv.MORPH_ELLIPSE,(3,3))
kernel_5 = cv.getStructuringElement(cv.MORPH_ELLIPSE,(5,5))
RESIZE_FACTOR = 2.0
MAJOR_DEFECT_THRESHOLD = 12.0 / RESIZE_FACTOR #6.0 5.0 12.0
ELLIPSE_RATIO_THRESHOLD = 5 # 2.0
MIN_AREA = 750# 500#702 # Menos de esto, no es un ratón.
MAX_AREA = 2800#2462#2.5 * MIN_AREA # Más de esto, no es un ratón.
BW_THRES = 12 # min B/W value for threshold
NUM_MICE = 3
RESIZE_FACTOR_X = 1 # Se autocompleta cuando lee la iagen y hace resize a new_size
RESIZE_FACTOR_Y = 1

N = 161 # number of images
#N = 242 # frames video_2 video_4
#N = 302 # framse video_3
#N = 1801 # framse video_5
#N = 1801 # framse video_6
# img size original: 640 x 480
normal_size = (480, 640)
new_size = (240, 320)

# TriMouse
#x_crop_min = int(normal_size[1]/19) #50
#x_crop_max = int(normal_size[1]/25) # 30
#y_crop_min = int(normal_size[0]/20) # 30
#y_crop_max = int(normal_size[0]/20) # 16 -- 40

# 3mice dataset
x_crop_min = int(normal_size[1]/5)#int(normal_size[1]/10) #50
x_crop_max = int(normal_size[1]/5)#int(normal_size[1]/20) # 30
y_crop_min = int(normal_size[0]/5)#int(normal_size[0]/16) # 30
y_crop_max = int(normal_size[0]/5) #int(normal_size[0]/20) # 16 -- 40

SEGMENT_COLORS = [(0,255,0),(0,255,255),(255,255,0),(255,0,255)]


In [32]:
df_tmp = pd.DataFrame()
df_times = pd.DataFrame()
nb = []
nm = []
if_overlaps = []
num_inter_points = []
times = []
coord = [None]*3
mus_x = np.full((3,N), None)
mus_y = np.full((3,N), None)
area = np.full((3,N), None)

cols = ["mus_1_x", "mus_1_y", "mus_1_area", "mus_2_x", "mus_2_y", "mus_2_area", "mus_3_x", "mus_3_y", "mus_3_area"]


In [33]:
# Lista para almacenar los nombres de los archivos
nombres_imagenes = []

# Recorrer todos los archivos en la carpeta
for archivo in os.listdir(ruta_carpeta):
    # Comprobar si el archivo es una imagen (puedes agregar más extensiones si es necesario)
    if archivo.endswith(('.png', '.jpg', '.jpeg', '.gif', '.bmp', '.tiff', '.webp')):
        archivo = "/"+archivo
        nombres_imagenes.append(archivo)

# Crear un DataFrame de Pandas con los nombres de las imágenes
df = pd.DataFrame(nombres_imagenes, columns=['img_path'])
img = ruta_carpeta + df['img_path'][2]



In [34]:
def read_image(i, ruta_carpeta, img_name):  
#    img_base = os.path.basename(img_name)
    img_base = os.path.splitext(img_name)[0]
    img_in = ruta_carpeta + img_name
    e1 = cv.getTickCount()
    frame = cv.imread(img_in)
    h,w,_ = frame.shape
    frame_o = frame[x_crop_min:(h-x_crop_max), y_crop_min:(w-y_crop_max)]
    resize_x = w / new_size[1]
    resize_y = h / new_size[0]
    frame = cv.resize(frame_o, (int(normal_size[1]/RESIZE_FACTOR), int(normal_size[0]/RESIZE_FACTOR))) # frame.shape/2
#     print(frame.shape) # 240 x 320
    e2 = cv.getTickCount()
    t_init_resize = (e2 - e1)/cv.getTickFrequency()
    return frame, resize_x, resize_y

In [35]:
def clean_image(frame, thres):
     frame_gray = cv.cvtColor(frame, cv.COLOR_BGR2GRAY)
     frame_norm = cv.normalize(frame_gray, None, alpha = 0, beta = 255, norm_type=cv.NORM_MINMAX)
     frame_blur = cv.medianBlur(frame_norm, 5)
     frame_blur = cv.equalizeHist(frame_blur)
     ret, frame_threshold = cv.threshold( frame_blur, thres, 255, cv.THRESH_BINARY_INV ) #+ cv.THRESH_OTSU )
#     frame_threshold = cv.adaptiveThreshold(frame_blur, 255, 
#                                               cv.ADAPTIVE_THRESH_GAUSSIAN_C, 
#                                               cv.THRESH_BINARY_INV, 11, 2)
     return frame_threshold     


In [36]:
def erode_image(frame):
     frame_erode = cv.morphologyEx(frame, cv.MORPH_DILATE, kernel_3, iterations = 4)  
     opening = cv.morphologyEx(frame_erode, cv.MORPH_OPEN, kernel_3, iterations = 6)


     return opening    

In [37]:
def detect_blobs(image, ruta_imagen, resize_x = 1, resize_y = 1):
#    contours, _ = cv.findContours(image, cv.RETR_EXTERNAL, cv.CHAIN_APPROX_SIMPLE)
    contours, _ = cv.findContours(image, cv.RETR_EXTERNAL, cv.CHAIN_APPROX_NONE)
    blobs = []
    for i, contour in enumerate(contours):
        area = cv.contourArea(contour)
        orig_x, orig_y, width, height = cv.boundingRect(contour)
        roi_image = image[orig_y:orig_y+height,orig_x:orig_x+width]
        perimeter = cv.arcLength(contour, True)
        blobs.append({
            "id" : i
            , "img_path": ruta_imagen
            , "resize_x": resize_x
            , "resize_y": resize_y
            , "contour" : contour
            , "origin" : (orig_x, orig_y)
            , "size" : (width, height)
            , "roi_image" : roi_image
            , "area" : area
            , "perimeter" : perimeter
            , "type": None
            , "ellipses": []
            ,"ellipse_area": []
            ,"hull": None
            ,"split_contours": []
        })
    return blobs
 

In [38]:
# split contour using convexity defects and a minimum of mice to detect
def split_blob_contours(blob, D, min_n_mice):
    D_min = 10
    D_max = 10e6
    min_n_mice = np.floor(2*(blob["area"] + 0.1 * blob["area"]) / (MIN_AREA + MAX_AREA))
    min_n_mice = min(min_n_mice, NUM_MICE) # como máximo es NUM_MICE
    min_n_mice = max(min_n_mice, 2) # como mínimo es 2
    print("expected N mice:", min_n_mice)    
    D_change_factor = 0.1
    max_n_points = (min_n_mice - 1) * 2 
    min_n_points = 2 
    contour = blob["contour"]
    contour = contour.squeeze()  # Elimina dimensiones extra si es necesario
    contour.tolist()
    blob["hull"] = cv.convexHull(contour)

    hull_idx = cv.convexHull(contour, returnPoints=False)
    defects = cv.convexityDefects(contour, hull_idx) 
    #defects are: [start_point, end_point, far_index, distance]
    # - start_point / end_point: closest points where convexHull and countour match
    # - far_index: index to the point of the contour with maximum distance to convexHull (between start and end)
    # - distance: between convexHull and contour point far_index.

    scores = np.zeros(len(blob["contour"]))

    intersections = []
    inter_points = []
    split_contours = []
    widths = []
    distances = []
    D_final = []
    n_points = 0
    if defects is not None :
#        print("defects:", defects)
        for i, defect in enumerate(np.squeeze(defects, 1)):#defects.shape[0]):
          s,e,f,dist= defect
          start = tuple(contour[s])
          end = tuple(contour[e])
          far = tuple(contour[f])
          width = np.linalg.norm(np.array(start) - np.array(end))
          scores[f] = dist*width
        n_points_old = 0
        n_iterations = 0
        while True:
            n_points = np.sum(scores >= D)
            print("n_points:", n_points)    
            if D < D_min or D > D_max: # por si acaso
              print("D out of range, stopping")
              break 
            # vamos incrementando  decrementando D hasta tener el numero de puntos deseado
            # para evitar bucles infinitos, vamos modificando el factor de cambio                
            else:
                if n_points > max_n_points: # too many points
                  D = D * (1 + D_change_factor)
                  print("Increasing D to:", D)
                elif n_points < min_n_points: # too few points
                  D = D * (1- D_change_factor)
                  print("Decreasing D to:", D)
                else:   # just right    
                    break
                if n_points == n_points_old:
                    D_change_factor *= 1.1 # increase change factor to escape oscillations
                elif n_points != n_points_old:
                    D_change_factor /= 1.1 # reduce change factor to avoid oscillations
            n_points_old = n_points
            n_iterations += 1
            if n_iterations > 100:
                print("Too many iterations, stopping")
                break
        print("Final D:", D)
        D_final.append(D)
        widths.append(width)
        distances.append(dist)
        D_change_factor = 0.1

        for i, defect in enumerate(np.squeeze(defects, 1)):#defects.shape[0]):
            s,e,f,dist= defect
            far = tuple(contour[f])
            if (scores[f] ) >= D: #MAJOR_DEFECT_THRESHOLD
               intersections.append(f)
               inter_points.append(far)
    intersections.sort()
    if n_points != len(intersections):
        print("Warning: n_points != len(intersections)")
        n_points = len(intersections)
        
    if (n_points == 0): # No se detectan defectos de convexidad
        print("No convexity defects detected.")
        split_contours = [contour]
    elif n_points == 1:
        index_plus = intersections[0] + len(contour)//2
        index_less = intersections[0] - len(contour)//2
        if intersections[0] < len(contour)//2:
            blob["segments"] = [
                contour[intersections[0]:index_plus]
                , np.vstack([contour[index_plus:],contour[:intersections[0]+1]])
    #            ,contour[midle_index:] + contour[:intersections[0]]
            ]
        else:
            blob["segments"] = [
                np.vstack([contour[intersections[0]:],contour[:index_less]])
                ,contour[index_less:intersections[0]+1]
                ]
        split_contours = [
            blob["segments"][0], blob["segments"][1]
        ]
    elif n_points == 2:
        if intersections[0] > intersections[1]:
            intersections[0], intersections[1] = intersections[1], intersections[0]
        blob["segments"] = [
                contour[intersections[0]:intersections[1]+1]
                , np.vstack([contour[intersections[1]:],contour[:intersections[0]+1]])
            ]
        blob["segments"] = [
            contour[intersections[0]:intersections[1]+1]
            , np.vstack([contour[intersections[1]:],contour[:intersections[0]+1]])
        ]
        split_contours = [
            blob["segments"][0], blob["segments"][1]
        ]
    elif n_points == 3:
        blob["segments"] = [
            contour[intersections[0]:intersections[1]+1]
            , contour[intersections[1]:intersections[2]+1]
            , np.vstack([contour[intersections[2]:],contour[:intersections[0]+1]])
        ]
        split_contours = [
            blob["segments"][0], blob["segments"][1], blob["segments"][2]
        ]
    elif n_points == 4:
        blob["segments"] = [
            contour[intersections[0]:intersections[1]+1]
            , contour[intersections[1]:intersections[2]+1]
            , contour[intersections[2]:intersections[3]+1]
            , np.vstack([contour[intersections[3]:],contour[:intersections[0]+1]])
        ]
        split_contours = [
            blob["segments"][0], blob["segments"][1],blob["segments"][2], blob["segments"][3]
        ]
    elif n_points == 5:
        blob["segments"] = [
            contour[intersections[0]:intersections[1]+1]
            , contour[intersections[1]:intersections[2]+1]
            , contour[intersections[2]:intersections[3]+1]
            , contour[intersections[2]:intersections[4]+1]
            , np.vstack([contour[intersections[4]:],contour[:intersections[0]+1]])
        ]
        split_contours = [
            blob["segments"][0], blob["segments"][1],blob["segments"][2], blob["segments"][3], blob["segments"][4]
        ]
    else :
       for i in range(0,len(contour),n_points):
          split_contour = contour[i:i + n_points]
          if len(split_contour)> 5:
             split_contours.append(split_contour)
    blob["split_contours"] = split_contours
    for c in split_contours:
        if len(c) >= 5:
            e = cv.fitEllipse(c)
            ea = (np.pi * (e[1][0]/2) * (e[1][1]/2  ))
            if MIN_AREA < ea < MAX_AREA and max(e[1][0], e[1][1]) / min(e[1][0],e[1][1]) < ELLIPSE_RATIO_THRESHOLD:
                blob["ellipse_area"].append(ea)
                blob["ellipses"].append(e) 
    blob["intersections"] = intersections    
    blob["inter_points"] = inter_points   
    blob["widths"] = widths
    blob["distances"] = distances
    blob["D_final"] = D_final
    
    return blob["ellipses"]

In [39]:
def process_blob(blob, num_mice = NUM_MICE-1):    
    if blob["area"] < MIN_AREA:
        blob["type"] = 0 # small blob
    elif MIN_AREA <= blob["area"] <= MAX_AREA:
        blob["type"] = 1 # medium blob
        e = cv.fitEllipse(blob["contour"])
        ea = (np.pi * (e[1][0]/2) * (e[1][1]/2  ))
        if MIN_AREA < ea < MAX_AREA and max(e[1][0], e[1][1]) / min(e[1][0],e[1][1]) < ELLIPSE_RATIO_THRESHOLD:
            blob["ellipses"].append(e)
            blob["ellipse_area"].append(ea)
#        print("blob ellipse:", blob["ellipses"][-1])
    elif blob["area"] > MAX_AREA:
        blob["type"] = 2 # large blob
        split_blob_contours(blob, 20000, num_mice) # Esto habría que ajustarlo al número de ratones que ya se han detectado


In [40]:
def write_results(image, blobs, thres, write_image_flag=True):
    SEGMENT_COLORS = [(0,255,0),(0,255,255),(255,255,0),(255,0,255)]
    output = image.copy()
    for blob in blobs:
        color = (0, 255, 0) if blob["type"] == 0 else (0, 255, 255) if blob["type"] == 1 else (255, 0, 0)
        cv.drawContours(output, [blob["contour"]], -1, color, 1)
        x, y = blob["origin"]
        w, h = blob["size"]
        cv.rectangle(output, (x, y), (x + w, y + h), color, 1)
        cv.putText(output, f'ID:{blob["id"]} A:{int(blob["area"])}', (x, y - 10), cv.FONT_HERSHEY_SIMPLEX, 0.5, color, 1)
        
        if blob["hull"] is not None:
            cv.drawContours(output, [blob["hull"]], -1, (255, 0, 255), 1)
#            cv.putText(output,str(blob["area"]),blob["origin"],, cv.FONT_HERSHEY_SIMPLEX, 0.6,(100,255,20),1)# Add character description
        
        for i, ellipse in enumerate(blob["ellipses"]):
#            if ellipse is not None:
                cv.ellipse(output, ellipse, (255,30,25), 1)
                cv.circle(output,(int(ellipse[0][0]),int(ellipse[0][1])), 3, (255,255,25))
                cv.putText(output,str(int(ellipse[0][0]))+","+str(int(ellipse[0][1])),(int(ellipse[0][0]-40),int(ellipse[0][1]+20)), cv.FONT_HERSHEY_SIMPLEX, 0.6,(100,255,20),1)# Add character description
                cv.putText(output,str(int(blob["ellipse_area"][i])),(int(ellipse[0][0]+10),int(ellipse[0][1]-5)), cv.FONT_HERSHEY_SIMPLEX, 0.6,(100,255,20),1)# Add character description


        
        if blob["split_contours"]:
            for n, split_contour in enumerate(blob["split_contours"]):
                cv.polylines(output, [split_contour], False, SEGMENT_COLORS[n%4], 2)   

        if "inter_points" in blob:
            for point in blob["inter_points"]:
                cv.circle(output, point, 3, (0, 0, 255))
    if write_image_flag and len(blobs) > 0:
        img_out = ruta_carpeta + "/no_labels_rev1/"+ str(thres) +"/"
        try:
            os.mkdir(img_out)
        except Exception as e:
            make__dir = "error"
        img_out += blob["img_path"].replace("/","")  
        print(img_out)
        cv.imwrite(img_out,output)


GROUND TRUTH


In [41]:
test_file = r'C:\Users\inges\OneDrive - UDIT\src\photoneu\dataset\deeplabcut\labeled-data-ordered\CollectedData_dlc.csv'
df_ground_truth = pd.DataFrame()
df_ground_truth = pd.read_csv(test_file)
df_ground_truth.head(20)

,scorer,dlc,dlc.1,dlc.2,dlc.3,dlc.4,dlc.5,dlc.6,dlc.7,dlc.8,...,dlc.62,dlc.63,dlc.64,dlc.65,dlc.66,dlc.67,dlc.68,dlc.69,dlc.70,dlc.71
0,individuals,mus1,mus1,mus1,mus1,mus1,mus1,mus1,mus1,mus1,...,mus3,mus3,mus3,mus3,mus3,mus3,mus3,mus3,mus3,mus3
1,bodyparts,snout,snout,leftear,leftear,rightear,rightear,shoulder,shoulder,spine1,...,spine4,spine4,tailbase,tailbase,tail1,tail1,tail2,tail2,tailend,tailend
2,coords,x,y,x,y,x,y,x,y,x,...,x,y,x,y,x,y,x,y,x,y
3,/img0038.png,577.0738982187231,251.0082247294906,575.3425399593766,273.5158821009952,596.5516786363714,272.65020297132196,586.1635290802924,281.7398338328911,586.1635290802924,...,NaN,NaN,NaN,NaN,NaN,NaN,127.01268085145685,332.08101456949316,162.72616092026604,331.8041658867892
4,/img0303.png,608.6823002384388,64.52077380905499,599.5926693768697,72.7840745922997,613.6402807083857,86.00535584549127,599.1795043377074,84.35269568884232,592.5688637111116,...,98.23955027247223,106.37117408591845,114.9010628721575,116.01731295942045,139.16256367520796,127.41729526446827,164.88560067121327,155.47879016920132,177.74711916921592,185.29412850548022
5,/img0502.png,391.6471699771854,235.93936498314736,411.53389946420856,212.30042238536515,384.51796506674316,206.67210271922653,401.0277027540831,199.5428978087843,404.7799158648422,...,101.12823717208212,99.68967706042972,116.58679305910454,109.58315282812407,141.9388247138213,125.6600509506274,166.05417189757628,147.9203714279397,188.62366349262902,178.21914096650363
6,/img0707.png,257.3440332490244,293.13160905008624,263.3644381053884,325.8138068417765,284.579198075433,312.33956740134283,279.70553700123355,326.9605506239411,288.592801313009,...,165.9229305514636,227.64039709550153,161.69290753146925,214.95032803551857,149.0028384714863,187.4551784055555,148.15683386748742,146.4239551116106,157.03988220947554,118.50580317964813
7,/img1138.png,280.5087701635501,316.5274748962508,286.18023465904184,290.0606405839558,260.1860223880378,299.9857034510664,270.58370729643946,283.44393200588206,265.85748688352965,...,574.6312641868317,137.98898003862422,566.3454782313877,156.0670584868657,549.0206530518229,185.44393596525813,531.3192012379199,209.17141392857508,507.2150966402645,233.2755185262304
8,/img1331.png,608.5274787976398,367.97728503505004,601.0230525761217,362.4740391392701,600.5227574946871,386.4882030481282,591.0171509474308,369.47817027935366,575.0077083415254,...,544.6293163108479,113.1035109963922,556.4080534247164,103.24224271501399,589.2789476959771,93.10705031470859,616.3974354697672,100.22907740681508,634.2025032000334,113.92528335317371
9,/img1501.png,442.834413410453,200.11096701580038,472.12863344437835,207.3620115786532,466.9078813591243,182.70846006495367,482.5701376148864,189.95950462780647,494.7518924804791,...,475.5193165256127,124.02018734728036,490.2791046451348,129.8554524177891,527.6934512736909,138.09347369380146,560.9887872642407,141.52598255880662,594.9706250277918,143.5854878778097


In [42]:
field = 'spine2'
df_clean = pd.DataFrame()
df_clean["img_path"] = df_ground_truth["scorer"]
df_clean.drop([0,1,2], inplace=True)
col = df_ground_truth.loc[:, (df_ground_truth.iloc[0] == 'mus1') & (df_ground_truth.iloc[1] == field) & (df_ground_truth.iloc[2] == 'x')]
df_clean["mus_1_x"] = col
col = df_ground_truth.loc[:, (df_ground_truth.iloc[0] == 'mus1') & (df_ground_truth.iloc[1] == field) & (df_ground_truth.iloc[2] == 'y')]
df_clean["mus_1_y"] = col
col = df_ground_truth.loc[:, (df_ground_truth.iloc[0] == 'mus2') & (df_ground_truth.iloc[1] == field) & (df_ground_truth.iloc[2] == 'x')]
df_clean["mus_2_x"] = col
col = df_ground_truth.loc[:, (df_ground_truth.iloc[0] == 'mus2') & (df_ground_truth.iloc[1] == field) & (df_ground_truth.iloc[2] == 'y')]
df_clean["mus_2_y"] = col
col = df_ground_truth.loc[:, (df_ground_truth.iloc[0] == 'mus3') & (df_ground_truth.iloc[1] == field) & (df_ground_truth.iloc[2] == 'x')]
df_clean["mus_3_x"] = col
col = df_ground_truth.loc[:, (df_ground_truth.iloc[0] == 'mus3') & (df_ground_truth.iloc[1] == field) & (df_ground_truth.iloc[2] == 'y')]
df_clean["mus_3_y"] = col
df_ground_truth = df_clean
df_ground_truth.head()


,img_path,mus_1_x,mus_1_y,mus_2_x,mus_2_y,mus_3_x,mus_3_y
3,/img0038.png,585.2978499506191,313.337122065965,96.94862418821859,320.6372283383545,45.34231945379248,398.52469841844044
4,/img0303.png,584.7187279670292,109.96892811690094,118.75671058567993,338.86136203326316,69.30113365196624,85.0327456687777
5,/img0502.png,410.0330142199049,165.77297981195258,527.4234560583649,422.39847716466613,69.59278316255637,81.44858111374327
6,/img0707.png,296.9066937337021,355.91583112359655,515.3951513434035,163.80427650680974,179.03600191344594,257.6735605374612
7,/img1138.png,258.2955342228739,251.30563319809525,338.64128124234094,220.5852005141814,586.6833164856594,107.48222265721671


OTRO GROUND_TRUTH MOVAI


In [43]:
ruta_moveai = r'C:\Users\inges\OneDrive - UDIT\src\photoneu\dataset\MovAl.v1i.tensorflow\train\ground_truth_centroids.csv'
df_moveai = pd.read_csv(ruta_moveai)
df_moveai.head()

,img_path,width,height,num_mice,mus_1_xmin,mus_1_ymin,mus_1_xmax,mus_1_ymax,mus_2_xmin,mus_2_ymin,...,mus_3_xmin,mus_3_ymin,mus_3_xmax,mus_3_ymax,mus_1_x,mus_1_y,mus_2_x,mus_2_y,mus_3_x,mus_3_y
0,240712_3w_148b_151b_154b_m_rec_1000model_mp4-0...,1920,1080,3,186,560,331,834,1171,173,...,1379,320,1639,491,258,697,1337,248,1509,405
1,240712_3w_148b_151b_154b_m_rec_1000model_mp4-0...,1920,1080,3,1311,183,1629,314,1393,367,...,189,574,329,830,1470,248,1512,491,259,702
2,240712_3w_148b_151b_154b_m_rec_1000model_mp4-0...,1920,1080,3,1321,170,1623,326,1477,431,...,186,569,331,834,1472,248,1565,595,258,701
3,240712_3w_148b_151b_154b_m_rec_1000model_mp4-0...,1920,1080,3,187,560,326,834,1489,594,...,1340,163,1599,343,256,697,1559,751,1469,253
4,240712_3w_148b_151b_154b_m_rec_1000model_mp4-0...,1920,1080,3,187,560,331,831,1340,174,...,1384,666,1636,877,259,695,1472,270,1510,771


In [44]:
# Construccion de un DF con todas las filas de un mismo img_path fusionadas
# y las columnas mus_i_x, mus_i_y, mus_i_type renumeradas consecutivamente
def merge_blob_rows(df_blob):
    # Detectar todas las columnas mus_i_x, mus_i_y, mus_i_type
    mus_cols = [c for c in df_blob.columns if c.startswith("mus_")]
    
    # Grupo por img_path
    grouped = df_blob.groupby("img_path")
    merged_rows = []
    for img_path, group in grouped:
        new_row = {"img_path": img_path.lstrip('/')}
        i_new = 1  # índice acumulativo para nueva numeración
        # recorrer cada fila del grupo
        for _, row in group.iterrows():
            new_row["resize_x"] = row["resize_x"]
            new_row["resize_y"] = row["resize_y"]
            # buscar todas las parejas mus_i_x/y/type válidas
            for i in range(1, 10):  # límite alto por seguridad
                x_col, y_col, t_col, s_col = f"mus_{i}_x", f"mus_{i}_y", f"mus_{i}_type", f"mus_{i}_area"
                if s_col not in row or pd.isna(row[s_col]):
                    continue
                # copiar datos a nuevas columnas consecutivas
                new_row[f"mus_{i_new}_x"] = row[x_col]
                new_row[f"mus_{i_new}_y"] = row[y_col]
                new_row[f"mus_{i_new}_type"] = row[t_col]
                new_row[f"mus_{i_new}_area"] = row[s_col]
                i_new += 1
                
        merged_rows.append(new_row)
    
    # reconstruir dataframe fusionado
    df_merged = pd.DataFrame(merged_rows)
    return df_merged

# --- ejemplo de uso ---
#df_merged = merge_blob_rows(df_blobs)
#print(df_merged)


RMSE Analysis


In [45]:
# Calculo de los RMSE entre las posiciones de los ratones detectados y las posiciones reales utilizando sklearn
# Como los ID de los ratones (mus_1, mus_2, mus_3) no coincide con los ID de la verdad (mus1, mus2, mus3),
# calculamos todas las combinaciones posibles de un ratón detectado con cada ratón de la verdad
# y nos quedamos con la que da menor RMSE para ese ratón detectado.
# Repetimos para el siguiente ratón detectado, pero sin repetir los ratones de la verdad ya asignados. 
# Guardar para cada ratón detectado su RMSE individual como un campo nuevo en el dataframe df_merged
# y luego representamos gráficamente esos RMSE individuales
from sklearn.metrics import mean_squared_error
import itertools

def clean_path_column(s):
    return (
        s.astype(str)              # convertir a string
         .str.strip()              # quitar espacios en blanco
         .str.replace('/', '', regex=False)  # eliminar '/'
    )

def calculate_individual_rmse(df_detected, df_truth):
    num_max_mice = 5
    individual_rmses = {f"mus_{i}_rmse": [] for i in range(1, num_max_mice)}
    rmses = []
    rmse_devs = []
    df_detected["img_path"] = clean_path_column(df_detected["img_path"])
    df_truth["img_path"] = clean_path_column(df_truth["img_path"])

    for _, row in df_detected.iterrows():
        img_path = row["img_path"]
#        truth_row = df_truth[df_truth["img_path"] == img_path]
        mask = df_truth["img_path"].isin([img_path])
        truth_row = df_truth[mask]
        if truth_row.empty:
            continue
        
        detected_positions = []
        for i in range(1, num_max_mice):  # suponiendo un máximo de 5 ratones por imagen
            x_col = f"mus_{i}_x"
            y_col = f"mus_{i}_y"
            if x_col in row and y_col in row and not pd.isna(row[x_col]) and not pd.isna(row[y_col]):
                detected_positions.append((i, (row[x_col], row[y_col])))
        
        truth_positions = []
        for i in range(1, num_max_mice):  # suponiendo un máximo de 5 ratones en la verdad
            x_col = f"mus_{i}_x"
            y_col = f"mus_{i}_y"
            if x_col in truth_row.columns and y_col in truth_row.columns and not pd.isna(truth_row.iloc[0][x_col]) and not pd.isna(truth_row.iloc[0][y_col]):
                truth_positions.append((i, (truth_row.iloc[0][x_col], truth_row.iloc[0][y_col])))
        
        n_detected = len(detected_positions)
        n_truth = len(truth_positions)
        
        if n_detected == 0 or n_truth == 0:
            continue
        rmse_array = []
        used_truth_indices = set()
        for det_idx, det_pos in detected_positions:
            min_rmse = float('inf')
            best_truth_idx = -1
            for truth_idx, truth_pos in truth_positions:
                if truth_idx in used_truth_indices:
                    continue
                rmse = mean_squared_error([truth_pos], [det_pos], squared=False)
                if rmse < min_rmse:
                    min_rmse = rmse
                    best_truth_idx = truth_idx
            if best_truth_idx != -1:
                used_truth_indices.add(best_truth_idx)
                individual_rmses[f"mus_{det_idx}_rmse"].append(min_rmse)
                rmse_array.append(min_rmse)

            else:
                individual_rmses[f"mus_{det_idx}_rmse"].append(None)
                # No hay ratón de la verdad disponible para este ratón detectado
        rmse_total = np.median(rmse_array)
        rmse_dev = np.std(rmse_array)
        rmses.append(rmse_total if rmse_total != float('inf') else None)
        rmse_devs.append(rmse_dev)

    # Añadir las columnas de RMSE individual al dataframe original
    for i in range(1, num_max_mice):
        rmse_col = f"mus_{i}_rmse"
        if rmse_col in individual_rmses:
            df_detected[rmse_col] = pd.Series(individual_rmses[rmse_col])

    #Añadir una columna "rmse" que sea la media de los mus_{i}_rmse válidos
    cols_rmse = [c for c in df_detected.columns if c.startswith('mus_') and c.endswith('_rmse')]
    df_detected["rmse_mean"] = df_detected[cols_rmse].mean(axis=1, skipna=True)
    df_detected["rmse_dev"] = df_detected[cols_rmse].std(axis=1, skipna=True)
     
    return df_detected

    

# --- ejemplo de uso ---
#df_merged = calculate_individual_rmse(df_merged, df_ground_truth)
#df_merged.head()





In [46]:
df_moveai.columns

Index(['img_path', 'width', 'height', 'num_mice', 'mus_1_xmin', 'mus_1_ymin',
       'mus_1_xmax', 'mus_1_ymax', 'mus_2_xmin', 'mus_2_ymin', 'mus_2_xmax',
       'mus_2_ymax', 'mus_3_xmin', 'mus_3_ymin', 'mus_3_xmax', 'mus_3_ymax',
       'mus_1_x', 'mus_1_y', 'mus_2_x', 'mus_2_y', 'mus_3_x', 'mus_3_y'],
      dtype='object')

Accuracy, Recall, F1 and RMSE statistics

In [47]:
# Calculamos PRecision, Recall y F1-Score globales para todos los mus_i juntos
# Y también el RMSE medio y su desviación estándar

from sklearn.metrics import precision_score, recall_score, f1_score

def clean_path_column(s):
    return (
        s.astype(str)              # convertir a string
         .str.strip()              # quitar espacios en blanco
         .str.replace('/', '', regex=False)  # eliminar '/'
    )

def calculate_overall_metrics(df_detected, df_truth, threshold=20):
    print("calculate_overall_metrics")
    num_max_mice = 5
    total_TP = total_FP = total_FN = 0
    df_detected["img_path"] = clean_path_column(df_detected["img_path"])
    df_truth["img_path"] = clean_path_column(df_truth["img_path"])

    for _, row in df_detected.iterrows():
        img_path = row["img_path"]
#        truth_row = df_truth[df_truth["img_path"] == img_path]
        mask = df_truth["img_path"].isin([img_path])
        truth_row = df_truth[mask]
        if truth_row.empty:
            continue
        detected_positions = []
        for i in range(1, num_max_mice):
            x_col = f"mus_{i}_x"
            y_col = f"mus_{i}_y"
            rmse_col = f"mus_{i}_rmse"
            threshold = np.sqrt(row["resize_x"] * row["resize_y"]) * 20
            if x_col in row and y_col in row and rmse_col in row and not pd.isna(row[x_col]) and not pd.isna(row[y_col]) and not pd.isna(row[rmse_col]):
                detected_positions.append((i, row[rmse_col], threshold))
        truth_positions = []
        for i in range(1, num_max_mice):
            x_col = f"mus_{i}_x"
            y_col = f"mus_{i}_y"
            if x_col in truth_row.columns and y_col in truth_row.columns and not pd.isna(truth_row.iloc[0][x_col]) and not pd.isna(truth_row.iloc[0][y_col]):
                truth_positions.append(i)
        
        detected_assigned = set()
        for det_idx, det_rmse, threshold in detected_positions:
            if det_rmse <= threshold:
                total_TP += 1
                detected_assigned.add(det_idx)
            else:
                total_FP += 1
        
        for truth_idx in truth_positions:
            if truth_idx not in detected_assigned:
                total_FN += 1
    rmse_mean = df_detected["rmse_mean"].mean() # medias de todo el df de las medias de cada raton
    rmse_std = df_detected["rmse_mean"].std()

    precision = total_TP / (total_TP + total_FP) if (total_TP + total_FP) > 0 else 0
    recall = total_TP / (total_TP + total_FN) if (total_TP + total_FN) > 0 else 0
    f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
    
    return { "RMSE_mean": rmse_mean, "RMSE_std":rmse_std,  "Precision": precision, "Recall": recall, "F1-Score": f1, "TP": total_TP, "FP": total_FP, "FN": total_FN}
# --- ejemplo de uso ---
#overall_metrics = calculate_overall_metrics(df_merged, df_ground_truth, threshold=20)
#print(f"Overall: Precision={overall_metrics['Precision']:.2f}, Recall={overall_metrics['Recall']:.2f}, F1-Score={overall_metrics['F1-Score']:.2f}, TP={overall_metrics['TP']}, FP={overall_metrics['FP']}, FN={overall_metrics['FN']}")


Pipeline

In [48]:
# Vamos analizar el efecto sobre precison, recall y F1-Score al variar el umbral BW_THRE de binarización
# desde 0 hasta 255 en pasos de 5
import warnings
warnings.filterwarnings("ignore")

scale_factor_x = RESIZE_FACTOR_X
scale_factor_y = RESIZE_FACTOR_Y

bw_thre = 12
print(f"Processing BW_THRE={bw_thre}")
total_blobs_local = []
for i, img in enumerate(df['img_path']):
    frame, resize_x, resize_y = read_image(i, ruta_carpeta, img)
    frame_clean = clean_image(frame, bw_thre)
    frame_clean = erode_image(frame_clean)
    blobs = detect_blobs(frame_clean, img, resize_x, resize_y)
    scale_factor_x = resize_x
    scale_factor_y = resize_y

    total_blobs_local.append(blobs)
    for b in blobs:
     print("blob area = %2d" % b["area"])
     if b["area"] < NUM_MICE*MAX_AREA:
        # asignamos el numero de ratones según la columna df_moveai a la que pertenece el blob
        #buscamos la columna de df_moveai que tiene el mismo img_path que el blob
#        num_mice = df_moveai.loc[df_moveai["filename"] == b["img_path"], "num_mice"].values[0]
        num_mice = NUM_MICE - 1
        process_blob(b, num_mice)
    write_results(frame, blobs, bw_thre, write_image_flag=True)
total_blobs_flat_local = [d for sublist in total_blobs_local for d in sublist]
df_blobs_local = pd.DataFrame(total_blobs_flat_local)
# print(df_blobs.head())
max_n = df_blobs_local["ellipses"].apply(len).max()
for i in range(max_n):
    df_blobs_local[f"mus_{i+1}_x"] = df_blobs_local["ellipses"].apply(
        lambda lst: lst[i][0][0] if i < len(lst) else None
    )
    df_blobs_local[f"mus_{i+1}_y"] = df_blobs_local["ellipses"].apply(
        lambda lst: lst[i][0][1] if i < len(lst) else None
    )
    df_blobs_local[f"mus_{i+1}_type"] = df_blobs_local["type"]
    df_blobs_local[f"mus_{i+1}_area"] = df_blobs_local["ellipse_area"].apply(
        lambda lst: lst[i] if i < len(lst) else None
    )   
df_merged = merge_blob_rows(df_blobs_local)
# df_merged = df_blobs

for i in range(1, 10):
    x_col = f"mus_{i}_x"
    y_col = f"mus_{i}_y"
    area_col = f"mus_{i}_area"
    if x_col in df_merged.columns:
        df_merged[x_col] = df_merged[x_col] * scale_factor_x
    if y_col in df_merged.columns:
        df_merged[y_col] = df_merged[y_col] * scale_factor_y
    if area_col in df_merged.columns:
        df_merged[area_col] = df_merged[area_col] * (scale_factor_x * scale_factor_y)
        


Processing BW_THRE=12
blob area = 645
blob area = 603
blob area = 279
blob area = 522
blob area = 231
blob area = 3772
expected N mice: 2.0
n_points: 3
Increasing D to: 22000.0
n_points: 3
Increasing D to: 24000.0
n_points: 3
Increasing D to: 26400.000000000004
n_points: 3
Increasing D to: 29304.000000000007
n_points: 3
Increasing D to: 32849.78400000001
n_points: 3
Increasing D to: 37222.090250400004
n_points: 3
Increasing D to: 42671.77648396107
n_points: 3
Increasing D to: 49544.10875847949
n_points: 3
Increasing D to: 58321.14984410756
n_points: 2
Final D: 58321.14984410756
C:\Users\inges\OneDrive - UDIT\src\photoneu\dataset\MovAl.v1i.tensorflow\train/no_labels_rev1/12/240513_3w_073b_076b_f_int_mask_100_jpg.rf.9b36f21841af6f56b01f1d317a977f1b.jpg
blob area = 650
blob area = 609
blob area = 279
blob area = 550
blob area = 219
blob area = 3772
expected N mice: 2.0
n_points: 3
Increasing D to: 22000.0
n_points: 3
Increasing D to: 24000.0
n_points: 3
Increasing D to: 26400.000000000004

In [49]:
df_merged.columns
df_merged.head()
df_merged.to_csv("df_detected.csv")

In [52]:
df_ground_truth_used = df_moveai
#len(df_ground_truth_used["img_path"]) - len(df_ground_truth_used["img_path"].drop_duplicates())
df_detected_matched = df_merged[df_merged['img_path'].isin(df_ground_truth_used['img_path'])].copy()
df_detected_matched = df_detected_matched.reset_index(drop=True)
df_detected_matched = calculate_individual_rmse(df_detected_matched, df_ground_truth_used)


In [51]:
#df_detected_matched.head()

In [53]:
#df_detected_matched.head()
#df_merged = calculate_overall_rmse(df_merged, df_ground_truth_used)
overall_metrics = calculate_overall_metrics(df_detected_matched, df_ground_truth_used)
results = [overall_metrics["RMSE_mean"], overall_metrics["RMSE_std"], bw_thre, overall_metrics["Precision"], overall_metrics["Recall"], overall_metrics["F1-Score"], overall_metrics["TP"], overall_metrics["FP"], overall_metrics["FN"]]

print(results)
#df_results = pd.DataFrame(results, columns=["RMSE_mean", "RMSE_std", "BW_THRE", "Precision", "Recall", "F1-Score"])
#df_results.head()




calculate_overall_metrics
[69.21815105520406, 28.867203395551225, 12, 0.8935064935064935, 0.546031746031746, 0.6778325123152709, 688, 82, 572]


Cross Validation BW threshold


In [ ]:
# Vamos analizar el efecto sobre precison, recall y F1-Score al variar el umbral BW_THRE de binarización
# desde 0 hasta 255 en pasos de 5
import warnings
warnings.filterwarnings("ignore")

df_ground_truth_used = df_moveai
    #len(df_ground_truth_used["img_path"]) - len(df_ground_truth_used["img_path"].drop_duplicates())

thresholds = range(0, 38, 1)
results = []
for bw_thre in thresholds:
    print(f"Processing BW_THRE={bw_thre}")
    total_blobs_local = []
#    for ruta_imagen in image_files:
    for i, img in enumerate(df['img_path']):
        frame, resize_x, resize_y = read_image(i, ruta_carpeta, img)
        frame_clean = clean_image(frame, bw_thre)
        frame_clean = erode_image(frame_clean)
        blobs = detect_blobs(frame_clean, img, resize_x, resize_y)
        total_blobs_local.append(blobs)
        for b in blobs:
            
#        print("blob area = %2d" % b["area"])
#        if b["area"] < 3000:
            process_blob(b, num_mice)
        write_results(frame, blobs, bw_thre, write_image_flag=True)
    total_blobs_flat_local = [d for sublist in total_blobs_local for d in sublist]
    df_blobs_local = pd.DataFrame(total_blobs_flat_local)
#    print(df_blobs.head())
    max_n = df_blobs_local["ellipses"].apply(len).max()
    for i in range(max_n):
        df_blobs_local[f"mus_{i+1}_x"] = df_blobs_local["ellipses"].apply(
            lambda lst: lst[i][0][0] if i < len(lst) else None
        )
        df_blobs_local[f"mus_{i+1}_y"] = df_blobs_local["ellipses"].apply(
            lambda lst: lst[i][0][1] if i < len(lst) else None
        )
        df_blobs_local[f"mus_{i+1}_type"] = df_blobs_local["type"]
        df_blobs_local[f"mus_{i+1}_area"] = df_blobs_local["ellipse_area"].apply(
            lambda lst: lst[i] if i < len(lst) else None
        )   
    df_merged = merge_blob_rows(df_blobs_local)
#    df_merged = df_blobs
    scale_factor = 2
    for i in range(1, 10):
        x_col = f"mus_{i}_x"
        y_col = f"mus_{i}_y"
        area_col = f"mus_{i}_area"
        if x_col in df_merged.columns:
            df_merged[x_col] = df_merged[x_col] * scale_factor
        if y_col in df_merged.columns:
            df_merged[y_col] = df_merged[y_col] * scale_factor
        if area_col in df_merged.columns:
            df_merged[area_col] = df_merged[area_col] * (scale_factor ** 2)

    df_detected_matched = df_merged[df_merged['img_path'].isin(df_ground_truth_used['img_path'])].copy()
    df_detected_matched = df_detected_matched.reset_index(drop=True)
    df_detected_matched = calculate_individual_rmse(df_detected_matched, df_ground_truth_used)
    overall_metrics = calculate_overall_metrics(df_detected_matched, df_ground_truth)
    results.append((overall_metrics["RMSE_mean"], overall_metrics["RMSE_std"], bw_thre, overall_metrics["Precision"], overall_metrics["Recall"], overall_metrics["F1-Score"]))

df_results = pd.DataFrame(results, columns=["RMSE_mean", "RMSE_std", "BW_THRE", "Precision", "Recall", "F1-Score"])
df_results.head()




Processing BW_THRE=0
C:\Users\inges\OneDrive - UDIT\src\photoneu\dataset\MovAl.v1i.tensorflow\train/no_labels_rev1/0/240513_3w_073b_076b_f_int_mask_100_jpg.rf.9b36f21841af6f56b01f1d317a977f1b.jpg
C:\Users\inges\OneDrive - UDIT\src\photoneu\dataset\MovAl.v1i.tensorflow\train/no_labels_rev1/0/240513_3w_073b_076b_f_int_mask_101_jpg.rf.e48935957ac57c6cb19e456db3f6a98b.jpg
C:\Users\inges\OneDrive - UDIT\src\photoneu\dataset\MovAl.v1i.tensorflow\train/no_labels_rev1/0/240513_3w_073b_076b_f_int_mask_102_jpg.rf.2b22bfa57251447256ec561be2db4ac2.jpg
C:\Users\inges\OneDrive - UDIT\src\photoneu\dataset\MovAl.v1i.tensorflow\train/no_labels_rev1/0/240513_3w_073b_076b_f_int_mask_103_jpg.rf.155994dfaf547deb45145b61da1522dc.jpg
C:\Users\inges\OneDrive - UDIT\src\photoneu\dataset\MovAl.v1i.tensorflow\train/no_labels_rev1/0/240513_3w_073b_076b_f_int_mask_104_jpg.rf.7d849781c2997d1ab2c6bc87a4988f9f.jpg
C:\Users\inges\OneDrive - UDIT\src\photoneu\dataset\MovAl.v1i.tensorflow\train/no_labels_rev1/0/240513_3

In [1]:
# Plot results
plt.figure(figsize=(10, 6))
plt.plot(df_results["BW_THRE"], df_results["Precision"], marker='o', label='Precision')
plt.plot(df_results["BW_THRE"], df_results["Recall"], marker='s', label='Recall')
plt.plot(df_results["BW_THRE"], df_results["F1-Score"], marker='^', label='F1-Score')
plt.xlabel('BW_THRE')   
plt.ylabel('Score')
plt.title('Precision, Recall and F1-Score vs BW_THRE')
plt.legend()    
plt.grid()
plt.xticks(range(0, 100, 5))
plt.show()


NameError: name 'plt' is not defined

In [ ]:
df_merged.columns
df_merged.describe()


,mus_1_x,mus_1_y,mus_1_type,mus_1_area,mus_2_x,mus_2_y,mus_2_type,mus_2_area,mus_3_x,mus_3_y,...,mus_1_rmse,mus_2_rmse,mus_3_rmse,mus_4_rmse,mus_5_rmse,mus_6_rmse,mus_7_rmse,mus_8_rmse,mus_9_rmse,rmse
count,154.000000,154.000000,154.000000,154.000000,130.000000,130.000000,130.000000,130.000000,59.000000,59.000000,...,106.000000,89.000000,41.000000,0.0,0.0,0.0,0.0,0.0,0.0,106.000000
mean,208.920613,351.262571,1.149351,8502.281547,320.537961,220.543587,1.561538,8705.113171,357.260609,150.765675,...,13.299647,16.716865,14.517368,NaN,NaN,NaN,NaN,NaN,NaN,16.984570
std,164.641416,94.010177,0.357597,971.651785,174.467267,91.942044,0.498118,1068.325357,172.455027,56.584471,...,35.234380,24.690976,3.665564,NaN,NaN,NaN,NaN,NaN,NaN,29.601967
min,31.014835,124.790222,1.000000,4386.774002,33.883736,23.744610,1.000000,2326.586016,27.115387,57.687737,...,2.074645,0.993936,4.579979,NaN,NaN,NaN,NaN,NaN,NaN,2.074645
25%,58.554615,313.578789,1.000000,7919.410160,152.687065,162.706089,1.000000,8521.889073,231.929008,135.647263,...,5.342236,8.037538,13.098292,NaN,NaN,NaN,NaN,NaN,NaN,9.558204
50%,160.874306,368.299194,1.000000,8723.637417,351.376190,184.111847,2.000000,8910.428367,411.492950,142.130859,...,7.532410,10.918405,14.658366,NaN,NaN,NaN,NaN,NaN,NaN,12.158124
75%,332.561775,427.632797,1.000000,9135.105424,498.571114,280.066338,2.000000,9308.141656,418.330261,166.219147,...,11.654613,18.233186,16.857651,NaN,NaN,NaN,NaN,NaN,NaN,14.804537
max,622.970825,451.979401,2.000000,9987.679997,612.710754,448.578003,2.000000,9895.611908,610.689880,331.908875,...,276.936644,167.408821,21.927637,NaN,NaN,NaN,NaN,NaN,NaN,222.041306
